# Correlated matching with per-edge regularization strengths

Correlated matching (`enable_correlations=True`) decodes in two passes. The first pass is ordinary minimum-weight perfect matching. Then, for every edge in that first-pass solution, the edges that are correlated with it (because some error mechanism, such as a Y error, flips both) are reweighted using the conditional probability that they also had an error. The second pass decodes again with these lower weights.

Reweighting with the full conditional probability can over-correct: below threshold, correlated decoding can lose some of the surface code's effective distance (see [arXiv:2610.26785](https://arxiv.org/abs/2610.26785)). PyMatching therefore lets you *regularize* the reweighting by multiplying each conditional probability by a strength $s$:

$$p_{\text{implied}} = \min\left(0.5,\; s \cdot P(\text{edge} \mid \text{matched edge})\right).$$

$s = 1$ is standard correlated matching, and $s < 1$ weakens the reweighting. There are two ways to set it:

- `global_reg_strength`: one strength for every edge.
- `reg_strengths`: one strength **per edge**. When an edge is reweighted, the strength of *that edge* (the one being reweighted, not the matched edge that caused it) is used.

Per-edge strengths are useful when the noise is not homogeneous across the device, for example when some qubits or couplers are much noisier than the rest. In this tutorial we:

1. build a surface code circuit where one half of the device is 10x noisier than the other,
2. show the different ways of passing `reg_strengths`,
3. tune one strength per region and compare against the best global strength on fresh data.

In [1]:
import numpy as np
import stim
from scipy.stats import binomtest

import pymatching

## A device with inhomogeneous noise

`stim.Circuit.generated` applies the same noise to every qubit. To mimic a device with a noisy region, we rescale the noise channels that touch a chosen set of qubits. A two-qubit channel is rescaled if either of its qubits is noisy.

In [2]:
NOISE_CHANNELS = {"DEPOLARIZE1": 1, "DEPOLARIZE2": 2, "X_ERROR": 1, "Z_ERROR": 1}


def scale_noise_on_qubits(circuit: stim.Circuit, noisy_qubits: set, factor: float) -> stim.Circuit:
    """Returns a copy of `circuit` where the noise channels acting on `noisy_qubits` are `factor` times stronger."""
    out = stim.Circuit()
    for inst in circuit:
        if isinstance(inst, stim.CircuitRepeatBlock):
            body = scale_noise_on_qubits(inst.body_copy(), noisy_qubits, factor)
            out.append(stim.CircuitRepeatBlock(inst.repeat_count, body))
        elif inst.name in NOISE_CHANNELS:
            arity = NOISE_CHANNELS[inst.name]
            targets = inst.targets_copy()
            p = inst.gate_args_copy()[0]
            quiet, noisy = [], []
            for k in range(0, len(targets), arity):
                group = targets[k:k + arity]
                (noisy if any(t.value in noisy_qubits for t in group) else quiet).extend(group)
            if quiet:
                out.append(inst.name, quiet, p)
            if noisy:
                out.append(inst.name, noisy, p * factor)
        else:
            out.append(inst)
    return out

We use a distance-5 rotated surface code memory experiment with $p = 0.001$ circuit-level noise, and make every qubit in the left half of the patch 10x noisier.

In [3]:
distance = 5
p = 0.001
noise_factor = 10

uniform_circuit = stim.Circuit.generated(
    "surface_code:rotated_memory_x",
    distance=distance,
    rounds=distance,
    after_clifford_depolarization=p,
    before_round_data_depolarization=p,
    before_measure_flip_probability=p,
    after_reset_flip_probability=p,
)

qubit_coords = uniform_circuit.get_final_qubit_coordinates()
x_values = [c[0] for c in qubit_coords.values()]
x_mid = (min(x_values) + max(x_values)) / 2
noisy_qubits = {q for q, c in qubit_coords.items() if c[0] < x_mid}

circuit = scale_noise_on_qubits(uniform_circuit, noisy_qubits, noise_factor)
print(f"{len(noisy_qubits)} of {len(qubit_coords)} qubits are in the noisy region")

22 of 49 qubits are in the noisy region


Correlated matching needs the detector error model to be loaded with `enable_correlations=True`, so that PyMatching keeps track of which edges are flipped together.

In [4]:
dem = circuit.detector_error_model(decompose_errors=True)
matching = pymatching.Matching.from_detector_error_model(dem, enable_correlations=True)
print(matching.num_edges, "edges")
matching.edges()[:3]

502 edges


[(0,
  3,
  {'fault_ids': set(),
   'weight': 3.9377826673290692,
   'error_probability': 0.01911873511075362}),
 (0,
  12,
  {'fault_ids': set(),
   'weight': 3.3161917299631716,
   'error_probability': 0.03501987560309133}),
 (0,
  None,
  {'fault_ids': {0},
   'weight': 3.8089606069523247,
   'error_probability': 0.021690311111111034})]

## Assigning a strength to each edge

Each edge is identified by the pair of nodes it connects, `(u, v)`, or `(u, None)` for an edge to the boundary, exactly as returned by `Matching.edges()`. Here the nodes are detectors, so we can use their coordinates to decide which region each edge belongs to. An edge between two detectors is assigned to the region of its midpoint.

On a real device you could instead group edges by their calibrated error probability (the `"error_probability"` entry of each edge's attributes), by the qubits involved, or tune them individually.

In [5]:
det_coords = dem.get_detector_coordinates()


def edge_region(u, v):
    xs = [det_coords[u][0]] if v is None else [det_coords[u][0], det_coords[v][0]]
    return "noisy" if np.mean(xs) < x_mid else "quiet"


regions = {(u, v): edge_region(u, v) for u, v, _ in matching.edges()}
print({r: list(regions.values()).count(r) for r in ("noisy", "quiet")})


def region_strengths(noisy_strength, quiet_strength):
    # One strength per edge, as a dict {(u, v): strength}
    return {edge: noisy_strength if region == "noisy" else quiet_strength for edge, region in regions.items()}

{'noisy': 213, 'quiet': 289}


### Three ways to pass `reg_strengths`

Per-edge strengths can be given as

1. a **dict** `{(u, v): strength}`. The node order in an edge does not matter, and edges left out of the dict get a strength of 1.0;
2. an **array** with one strength per edge, in the order of `Matching.edges()`;
3. **stored** on the `Matching` object with `set_reg_strengths`, after which every correlated decode uses them unless strengths are passed to that call.

They all give the same predictions:

In [6]:
dets, obs = circuit.compile_detector_sampler(seed=1).sample(10_000, separate_observables=True)
strengths = region_strengths(1.0, 0.5)

# 1. A dict passed to the decode call
pred_dict = matching.decode_batch(dets, enable_correlations=True, reg_strengths=strengths)

# 2. An array in `matching.edges()` order
strengths_array = np.array([strengths[(u, v)] for u, v, _ in matching.edges()])
pred_array = matching.decode_batch(dets, enable_correlations=True, reg_strengths=strengths_array)

# 3. Stored on the Matching object
matching.set_reg_strengths(strengths)
pred_stored = matching.decode_batch(dets, enable_correlations=True)

print(np.array_equal(pred_dict, pred_array), np.array_equal(pred_dict, pred_stored))

True True


The stored strengths can be read back with `Matching.reg_strengths` (a dict over all edges). Strengths passed to a decode call override the stored ones for that call only. `set_reg_strengths(None)` resets every edge to 1.0, and `set_global_reg_strength` replaces per-edge strengths with a single strength.

In [7]:
u, v = next(edge for edge, region in regions.items() if region == "quiet")
print("stored strength of quiet edge", (u, v), "is", matching.reg_strengths[(u, v)])

# Only change one edge for this call; every other edge gets 1.0
matching.decode_batch(dets, enable_correlations=True, reg_strengths={(v, u): 0.3})

matching.set_reg_strengths(None)
print(matching.reg_strengths, matching.global_reg_strength)

stored strength of quiet edge (3, 6) is 0.5
None 1.0


A few rules to keep in mind:

- `global_reg_strength` and `reg_strengths` cannot be combined (in a decode call or in the setters). Doing so raises a `ValueError`.
- Strengths must be positive, and only affect decoding when `enable_correlations=True`.
- If you add edges to the graph after `set_reg_strengths`, call it again: decoding raises an error otherwise.
- For many shots, prefer `decode_batch` with stored strengths. A dict passed to `decode` is converted to an array on every call.

## Tuning the strengths

Which strengths work best depends on the code, the noise and the physical error rate, so they are best chosen from data. We sample a tuning set of shots and compare:

- uncorrelated matching,
- correlated matching with one global strength,
- correlated matching with one strength for the noisy region and another for the quiet region.

In [8]:
def count_errors(dets, obs, **kwargs):
    predictions = matching.decode_batch(dets, **kwargs)
    return int(np.sum(np.any(predictions != obs, axis=1)))


num_tuning_shots = 300_000
tune_dets, tune_obs = circuit.compile_detector_sampler(seed=2).sample(num_tuning_shots, separate_observables=True)
print("uncorrelated matching:", count_errors(tune_dets, tune_obs), "errors")

grid = [0.25, 0.5, 0.75, 1.0]
results = {}
for s_noisy in grid:
    for s_quiet in grid:
        results[s_noisy, s_quiet] = count_errors(
            tune_dets, tune_obs, enable_correlations=True, reg_strengths=region_strengths(s_noisy, s_quiet)
        )

print(f"\nlogical errors out of {num_tuning_shots} shots (rows: noisy region, columns: quiet region)")
print("s_noisy \\ s_quiet" + "".join(f"{s:>8}" for s in grid))
for s_noisy in grid:
    print(f"{s_noisy:>17}" + "".join(f"{results[s_noisy, s_quiet]:>8}" for s_quiet in grid))

uncorrelated matching: 1376 errors



logical errors out of 300000 shots (rows: noisy region, columns: quiet region)
s_noisy \ s_quiet    0.25     0.5    0.75     1.0
             0.25    1028    1029    1046    1085
              0.5     995     991    1009    1041
             0.75     979     976     984    1016
              1.0     982     956     971     998


The diagonal of the table is the same as using `global_reg_strength`. Every other cell uses a different strength in each region.

Picking the best cell of a table and reporting its error count is optimistically biased, because some of its advantage is just sampling noise in the tuning set. So we pick the best global strength (the best diagonal entry) and the best per-region strengths on the tuning data, then compare them on a **fresh** set of shots.

In [9]:
best_global = min(grid, key=lambda s: results[s, s])
best_per_region = min(results, key=results.get)
print("best global strength:", best_global)
print("best (noisy, quiet) strengths:", best_per_region)

best global strength: 0.75
best (noisy, quiet) strengths: (1.0, 0.5)


### Comparing two decoders on the same shots

Because both decoders decode the same shots, the most sensitive comparison is a *paired* one: count the shots that only one of the two decoders gets wrong. If the two decoders were equally good, each of these disagreements would be equally likely to go either way, which a binomial (sign) test checks.

In [10]:
num_eval_shots = 1_000_000
eval_dets, eval_obs = circuit.compile_detector_sampler(seed=3).sample(num_eval_shots, separate_observables=True)


def failures(**kwargs):
    return np.any(matching.decode_batch(eval_dets, **kwargs) != eval_obs, axis=1)


fail_uncorrelated = failures()
fail_global = failures(enable_correlations=True, global_reg_strength=best_global)
fail_region = failures(enable_correlations=True, reg_strengths=region_strengths(*best_per_region))

for name, fail in [
    ("uncorrelated", fail_uncorrelated),
    (f"global strength {best_global}", fail_global),
    (f"per-region strengths {best_per_region}", fail_region),
]:
    rate = fail.mean()
    stderr = np.sqrt(rate * (1 - rate) / num_eval_shots)
    print(f"{name:>36}: logical error rate {rate:.2e} +/- {stderr:.1e}")

only_global = int(np.sum(fail_global & ~fail_region))
only_region = int(np.sum(fail_region & ~fail_global))
print(f"\nshots only the global-strength decoder gets wrong:     {only_global}")
print(f"shots only the per-region-strength decoder gets wrong: {only_region}")
print(f"sign test p-value: {binomtest(only_region, only_global + only_region).pvalue:.1e}")

                        uncorrelated: logical error rate 4.45e-03 +/- 6.7e-05
                global strength 0.75: logical error rate 3.25e-03 +/- 5.7e-05
     per-region strengths (1.0, 0.5): logical error rate 3.15e-03 +/- 5.6e-05

shots only the global-strength decoder gets wrong:     361
shots only the per-region-strength decoder gets wrong: 268
sign test p-value: 2.4e-04


In this example:

- Correlated matching already improves substantially on uncorrelated matching.
- The tuning table shows that the two regions prefer different strengths. Edges in the noisy region, whose physical error rate is 10x higher, do best with the full reweighting ($s = 1$). Edges in the quiet region do best with a weaker one ($s \approx 0.5$). This agrees with the observation that regularizing the reweighting helps most at low physical error rates.
- No single global strength can satisfy both regions, so per-region strengths beat the best global strength on the fresh evaluation shots. The improvement in the logical error rate is modest, but the paired comparison shows it is clearly significant.

The exact numbers depend on the random samples (stim only guarantees identical samples for a fixed seed with the same stim version and machine), but the pattern is robust. How much per-edge strengths help depends on how inhomogeneous the noise is, so it is worth repeating this tuning with a detector error model that reflects your own device.

## Using the tuned strengths

Once you have chosen the strengths, store them on the `Matching` object so that every correlated decode uses them:

In [11]:
matching.set_reg_strengths(region_strengths(*best_per_region))
predictions = matching.decode_batch(eval_dets[:5], enable_correlations=True)
predictions

array([[0],
       [0],
       [1],
       [0],
       [0]], dtype=uint8)